In [ ]:
# =============================================================================
# MANUAL PIPELINE: Raw Configuration Extraction -> Template Generation -> User Fill-In
# =============================================================================
# This notebook demonstrates the manual calibration workflow:
# 1. Read raw file configurations 
# 2. Identify unique channel configurations based on matching parameters
# 3. Generate calibration template files with null values for user to fill in
# 4. Generate mapping file (derived directly from raw data, no algorithm needed)
# 5. User fills in calibration values
# 6. Load filled templates 
# =============================================================================

import sys
from pathlib import Path
import datetime

# Add the package source directory to path so calibration_library is importable
# (not needed if the package is installed via pip install -e .)
sys.path.insert(0, str(Path("../src").resolve()))

import json
import yaml

# Raw file reading
from calibration_library.simrad_reader.raw_reader import SimradFileReader
from calibration_library.raw_reader_api import (
    detect_instrument_type,
    extract_ek60_file_config,
    extract_ek80_file_config,
    read_ek80_xml_as_dict,
    _pretty_dict,
    save_yaml,
    extract_unique_channels,
)

# Calibration template generation & validation
from calibration_library.standardized_file_lib import (
    create_calibration_template,
    save_template_with_comments,
    save_multi_channel_config_with_comments,
    load_calibration_templates,
    check_required_fields,
    calibration_key_to_filename,
    build_short_filename_map,
    remap_to_short_keys,
    print_short_key_summary,
)

# Mapping
from calibration_library.mapping_algorithm import (
    build_mapping_from_raw_configs,
    get_calibration,
)

print("All imports successful!")

All imports successful!


In [ ]:
# =============================================================================
# CONFIGURATION - Define Input and Output Paths
# =============================================================================

# Record author - the individual running this pipeline and generating the records
RECORD_AUTHOR = "Placeholder Author"

# Filename style for single-channel calibration template files:
#   False -> long filenames derived from the full calibration key
SHORT_FILENAMES = False

# User-provided calibration date (required for template generation)
CALIBRATION_YEAR = 2026  
CALIBRATION_MONTH = 2    
CALIBRATION_DAY = 18    

CALIBRATION_DATE = f"{CALIBRATION_YEAR:04d}-{CALIBRATION_MONTH:02d}-{CALIBRATION_DAY:02d}"

# Input folder - same as full pipeline
# RAW_INPUT_FOLDER = Path("./example_data/ek60_raw_file_input_folder")
RAW_INPUT_FOLDER = Path("./example_data/ek80_CW_raw_file_input_folder")


# Output folder with subfolders for organization
OUTPUT_BASE = Path("./Manual_Pipeline_Outputs")

# Create output subdirectories
RAW_CONFIGS_OUTPUT = OUTPUT_BASE / "raw_file_configs"
MAPPING_OUTPUT = OUTPUT_BASE / "mapping_files"
SINGLE_CAL_OUTPUT = OUTPUT_BASE / "Single_Channel_Calibration_Templates"

# Create all output directories
for folder in [RAW_CONFIGS_OUTPUT, MAPPING_OUTPUT, SINGLE_CAL_OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)

# Output file paths
RAW_CONFIGS_PATH = RAW_CONFIGS_OUTPUT / "raw_file_configs.yaml"
MAPPING_PATH = MAPPING_OUTPUT / "channel_to_calibration_mapping.yaml"
CALIBRATION_DICT_PATH = MAPPING_OUTPUT / "calibration_configurations.yaml"

print(f"Record author: {RECORD_AUTHOR}")
print(f"Short filenames: {SHORT_FILENAMES}")
print(f"\nInput folders:")
print(f"  Raw files: {RAW_INPUT_FOLDER}")
print(f"\nOutput folders:")
print(f"  Raw configs:              {RAW_CONFIGS_OUTPUT}")
print(f"  Mapping files:            {MAPPING_OUTPUT}")
print(f"  Calibration templates:    {SINGLE_CAL_OUTPUT}")
print(f"\nCalibration date: {CALIBRATION_DATE}")

Record author: Placeholder Author
Short filenames: False

Input folders:
  Raw files: ek80_CW_raw_file_input_folder

Output folders:
  Raw configs:              Manual_Pipeline_Outputs\raw_file_configs
  Mapping files:            Manual_Pipeline_Outputs\mapping_files
  Calibration templates:    Manual_Pipeline_Outputs\Single_Channel_Calibration_Templates

Calibration date: 2026-02-18


In [3]:
# =============================================================================
# STEP 1: READ RAW FILE CONFIGURATIONS
# =============================================================================
# Process all .raw files in the input folder and extract channel configurations
# (This step is identical to the full pipeline)

# Auto-discover all .raw files in the directory
raw_files = sorted(RAW_INPUT_FOLDER.glob("*.raw"))
# NOTE: may want to sort by time stamps

if not raw_files:
    raise FileNotFoundError(f"No .raw files found in: {RAW_INPUT_FOLDER}")

print(f"Found {len(raw_files)} raw files in {RAW_INPUT_FOLDER}")
for f in raw_files:
    print(f"  - {f.name}")

# Process all raw files and collect configurations
file_configs = []
frequencies_set = set()

for raw_path in raw_files:
    # Auto-detect instrument type
    instrument = detect_instrument_type(raw_path)
    
    print("=" * 80)
    print(f"File: {raw_path.name}")
    print(f"Instrument (detected): {instrument}")
    
    if instrument == "UNKNOWN":
        print("  WARNING: Could not detect instrument type, skipping file")
        continue
    
    reader = SimradFileReader(instrument)
    reader.process_file(str(raw_path))
    
    print(f"File format (from reader): {reader.file_format}")

    metadata = json.loads(reader.metadata) if reader.metadata else None
    if not metadata:
        print("Header parameters (FILE_PARAMETERS): <none>")

    if reader.errors:
        _pretty_dict("Read errors:", reader.errors)

    if instrument == "EK80":
        ek80_dict = read_ek80_xml_as_dict(raw_path)
        file_config = extract_ek80_file_config(raw_path, ek80_dict, metadata)
        file_config["file_format"] = reader.file_format
        file_config["instrument"] = instrument

    elif instrument == "EK60":
        file_config = extract_ek60_file_config(raw_path, reader, metadata)
        file_config["instrument"] = instrument
    
    file_configs.append(file_config)
    
    # Collect unique frequencies for summary
    for ch in file_config.get("channels", []):
        if "frequency" in ch:
            frequencies_set.add(ch["frequency"])
    
    # Display GPS summary
    gps = file_config.get("gps_data", {})
    print(f"\n--- GPS Summary ---")
    print(f"  NMEA datagrams found: {gps.get('nmea_count', 0)}")
    print(f"  Valid GPS fixes: {gps.get('valid_gps_count', 0)}")
    if gps.get("first_gps"):
        fg = gps["first_gps"]
        print(f"  First GPS: {fg['latitude']:.6f}, {fg['longitude']:.6f}")

print("\n" + "=" * 80)
print(f"SUMMARY: Processed {len(file_configs)} files")
print(f"Unique frequencies found: {sorted(frequencies_set)} Hz")
print("=" * 80)

# Save raw file configurations to YAML
save_yaml(file_configs, RAW_CONFIGS_PATH)
print(f"\n Saved raw file configurations to: {RAW_CONFIGS_PATH}")

Found 1 raw files in ek80_CW_raw_file_input_folder
  - D20241004-T061522.raw
File: D20241004-T061522.raw
Instrument (detected): EK80
File format (from reader): EK80

--- GPS Summary ---
  NMEA datagrams found: 86438
  Valid GPS fixes: 3325
  First GPS: 40.650391, -70.763870

SUMMARY: Processed 1 files
Unique frequencies found: [18000.0, 38000.0, 70000.0, 120000.0, 200000.0] Hz

 Saved raw file configurations to: Manual_Pipeline_Outputs\raw_file_configs\raw_file_configs.yaml


In [4]:
# =============================================================================
# STEP 2: IDENTIFY UNIQUE CHANNEL CONFIGURATIONS
# =============================================================================
# The matching parameters are: 
# - transceiver_id, transducer_model, transducer_serial_number, pulse_form,
# - frequency_start, frequency_end, transmit_power, transmit_duration_nominal
#
# Each unique combination of these parameters gets its own calibration template.
# Multiple raw files/channels sharing the same configuration will use the same calibration.
#
# Functions used: extract_unique_channels (from raw_reader_api)

# Extract unique channels
unique_channels = extract_unique_channels(file_configs, CALIBRATION_DATE)

print(f"Found {len(unique_channels)} unique channel configurations:")
print("=" * 80)
for key, channel in unique_channels.items():
    print(f"\nChannel: {channel.get('channel_id')}")
    print(f"  Key: {key}")
    print(f"  Transceiver ID: {channel.get('transceiver_id')}")
    print(f"  Transducer model: {channel.get('transducer_model')}")
    print(f"  Transducer serial number: {channel.get('transducer_serial_number', 'N/A')}")
    print(f"  Pulse form: {channel.get('pulse_form')}")
    print(f"  Frequency: {channel.get('frequency_start')} - {channel.get('frequency_end')} Hz")
    print(f"  Transmit power: {channel.get('transmit_power')} W")
    print(f"  Transmit duration: {channel.get('transmit_duration_nominal')} s")

Found 5 unique channel configurations:

Channel: WBT 400479-15 ES18_ES
  Key: 2026-02-18__WBT 400479-15 ES18_ES__NoSN__0__0.001024__1000.0__18000.0__18000.0
  Transceiver ID: 400479
  Transducer model: ES18
  Transducer serial number: None
  Pulse form: 0
  Frequency: 18000.0 - 18000.0 Hz
  Transmit power: 1000.0 W
  Transmit duration: 0.001024 s

Channel: WBT 400528-15 ES38-7_ES
  Key: 2026-02-18__WBT 400528-15 ES38-7_ES__447__0__0.001024__1000.0__38000.0__38000.0
  Transceiver ID: 400528
  Transducer model: ES38-7
  Transducer serial number: 447
  Pulse form: 0
  Frequency: 38000.0 - 38000.0 Hz
  Transmit power: 1000.0 W
  Transmit duration: 0.001024 s

Channel: WBT 400503-15 ES70-7C_ES
  Key: 2026-02-18__WBT 400503-15 ES70-7C_ES__NoSN__0__0.001024__750.0__70000.0__70000.0
  Transceiver ID: 400503
  Transducer model: ES70-7C
  Transducer serial number: None
  Pulse form: 0
  Frequency: 70000.0 - 70000.0 Hz
  Transmit power: 750.0 W
  Transmit duration: 0.001024 s

Channel: WBT 400517

In [5]:
# =============================================================================
# STEP 3: GENERATE CALIBRATION TEMPLATE FILES
# =============================================================================
# Create template files with null values for the user to fill in.
# Each unique channel configuration gets its own template file.
#
# Functions used: create_calibration_template, save_template_with_comments
#   (from standardized_file_lib)

# Generate a shared timestamp for all templates in this batch
batch_timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat()

# Build filename mapping (short or long)
if SHORT_FILENAMES:
    filename_map = build_short_filename_map(unique_channels, calibration_date=CALIBRATION_DATE)
else:
    filename_map = {k: calibration_key_to_filename(k) for k in unique_channels}

# Generate templates for each unique channel (keyed by long internal key)
calibration_templates = {}

for channel_key, channel in unique_channels.items():
    template = create_calibration_template(channel, CALIBRATION_DATE)
    # Set record_created for this batch
    template['record_created'] = batch_timestamp
    # Set record_author from configuration
    template['record_author'] = RECORD_AUTHOR
    calibration_templates[channel_key] = template
    
    # Save individual template file with comments
    safe_name = filename_map[channel_key]
    template_file = SINGLE_CAL_OUTPUT / f"{safe_name}.yml"
    save_template_with_comments(template, template_file, CALIBRATION_DATE)

# When using short filenames, remap the calibration_templates dict to use
# short keys as the canonical identifiers going forward.
if SHORT_FILENAMES:
    calibration_templates = {
        filename_map[k]: v for k, v in calibration_templates.items()
    }
    print_short_key_summary(filename_map, {k: unique_channels[k] for k in filename_map})

print(f"\n Generated {len(calibration_templates)} calibration template files")
print(f"   Output directory: {SINGLE_CAL_OUTPUT}")
print(f"   Filename style: {'short' if SHORT_FILENAMES else 'long'}")
print(f"   All templates share record_created: {batch_timestamp}")
print(f"   Record author: {RECORD_AUTHOR}")
print("\nTemplate files created:")
for template_file in sorted(SINGLE_CAL_OUTPUT.glob("*.yml")):
    print(f"  - {template_file.name}")


 Generated 5 calibration template files
   Output directory: Manual_Pipeline_Outputs\Single_Channel_Calibration_Templates
   Filename style: long
   All templates share record_created: 2026-03-12T21:46:31.701435+00:00
   Record author: Placeholder Author

Template files created:
  - 2026-02-18__WBT 400479-15 ES18_ES__NoSN__0__0.001024__1000.0__18000.0__18000.0.yml
  - 2026-02-18__WBT 400503-15 ES70-7C_ES__NoSN__0__0.001024__750.0__70000.0__70000.0.yml
  - 2026-02-18__WBT 400509-15 ES200-7C_ES__NoSN__0__0.001024__150.0__200000.0__200000.0.yml
  - 2026-02-18__WBT 400517-15 ES120-7C_ES__NoSN__0__0.001024__250.0__120000.0__120000.0.yml
  - 2026-02-18__WBT 400528-15 ES38-7_ES__447__0__0.001024__1000.0__38000.0__38000.0.yml


In [6]:
# =============================================================================
# STEP 4: GENERATE MAPPING FILE
# =============================================================================
# Since the calibration templates are derived directly from raw file channels,
# the mapping is deterministic - no matching algorithm needed.
# Each raw file channel maps to the calibration template with the same key.
#
# Function used: build_mapping_from_raw_configs (from mapping_algorithm)

# Build the mapping file (uses long internal keys)
mapping_dict = build_mapping_from_raw_configs(file_configs, CALIBRATION_DATE)

# When using short filenames, remap the mapping values to short identifiers
if SHORT_FILENAMES:
    filename_map = build_short_filename_map(unique_channels, calibration_date=CALIBRATION_DATE)
    for fname, channels in mapping_dict.items():
        for channel_id in channels:
            long_key = channels[channel_id]
            channels[channel_id] = filename_map.get(long_key, long_key)

# Save mapping file
with open(MAPPING_PATH, 'w') as f:
    yaml.dump(mapping_dict, f, default_flow_style=False, sort_keys=False)

print(f" Saved mapping file to: {MAPPING_PATH}")
print("\nMapping dictionary preview:")
print("=" * 80)
for filename, channels in mapping_dict.items():
    print(f"\n{filename}:")
    for channel_id, cal_key in channels.items():
        print(f"  {channel_id}")
        print(f"    -> {cal_key}")

 Saved mapping file to: Manual_Pipeline_Outputs\mapping_files\channel_to_calibration_mapping.yaml

Mapping dictionary preview:

D20241004-T061522.raw:
  WBT 400479-15 ES18_ES
    -> 2026-02-18__WBT 400479-15 ES18_ES__NoSN__0__0.001024__1000.0__18000.0__18000.0
  WBT 400528-15 ES38-7_ES
    -> 2026-02-18__WBT 400528-15 ES38-7_ES__447__0__0.001024__1000.0__38000.0__38000.0
  WBT 400503-15 ES70-7C_ES
    -> 2026-02-18__WBT 400503-15 ES70-7C_ES__NoSN__0__0.001024__750.0__70000.0__70000.0
  WBT 400517-15 ES120-7C_ES
    -> 2026-02-18__WBT 400517-15 ES120-7C_ES__NoSN__0__0.001024__250.0__120000.0__120000.0
  WBT 400509-15 ES200-7C_ES
    -> 2026-02-18__WBT 400509-15 ES200-7C_ES__NoSN__0__0.001024__150.0__200000.0__200000.0


In [7]:
# =============================================================================
# STEP 4 (continued): SAVE CALIBRATION CONFIGURATIONS FILE
# =============================================================================
# The calibration_configurations.yaml file is an alternative representation
# of the same data as the individual template files. Users can choose to 
# edit either one - both represent the same data structure.
#
# Function used: save_multi_channel_config_with_comments (from standardized_file_lib)

# Save calibration configurations with comments
save_multi_channel_config_with_comments(calibration_templates, CALIBRATION_DICT_PATH)

print(f" Saved calibration configurations to: {CALIBRATION_DICT_PATH}")

 Saved calibration configurations to: Manual_Pipeline_Outputs\mapping_files\calibration_configurations.yaml


# STEP 5: USER ACTION REQUIRED - FILL IN CALIBRATION VALUES

## STOP HERE AND FILL IN YOUR CALIBRATION DATA

The template files have been generated. Before proceeding to Step 6, you must fill in your calibration values.

### Option A: Edit Individual Template Files
Navigate to the folder:
```
Manual_Pipeline_Outputs/Single_Channel_Calibration_Templates/
```
Each file contains one channel's calibration template. Fill in the values marked as `null`.
**Note:** The calibration date (`calibration_date`) is already set from the configuration at the start of the notebook. You do not need to fill in this field.

### Option B: Edit the Consolidated File
Edit the single file:
```
Manual_Pipeline_Outputs/mapping_files/calibration_configurations.yaml
```
This contains all calibration templates in one file.

---

## Required Fields to Fill In:
| Field | Description | Example |
|-------|-------------|---------|
| `gain_correction` | Transducer gain [dB] | `[24.06]` |
| `sa_correction` | Sa correction [dB] | `[-0.68]` |
| `equivalent_beam_angle` | Equivalent beam angle [dB] | `-20.6` |
| `absorption_indicative` | Absorption coefficient [dB/m] | `0.0072` |
| `sound_speed_indicative` | Sound speed [m/s] | `1522.6` |

## Optional Fields:
- `beamwidth_transmit/receive_major/minor` - Beam angles
- `echoangle_major/minor` - Angle offsets
- `echoangle_major/minor_sensitivity` - Angle sensitivities
- `sample_interval`, `transmit_bandwidth`, etc.
- `calibration_comments`, `sphere_diameter`, `sphere_material`

---

**When you have filled in the calibration values, continue to Step 6 below.**

In [8]:
# =============================================================================
# STEP 6: LOAD FILLED CALIBRATION TEMPLATES AND VALIDATE
# =============================================================================
# After the user has filled in calibration values, load the templates
# and check for completeness. This step also regenerates the 
# calibration_configurations.yaml file from the individual template files.
#
# Functions used: load_calibration_templates, check_required_fields
#   (from standardized_file_lib)

# Load the filled templates from individual files
loaded_templates = load_calibration_templates(SINGLE_CAL_OUTPUT)

print(f"Loaded {len(loaded_templates)} calibration templates")
print("=" * 80)

# Check each template for completeness
all_complete = True
for template_key, template in loaded_templates.items():
    unfilled = check_required_fields(template)
    channel = template.get('channel', 'Unknown')
    
    if unfilled:
        all_complete = False
        print(f"\n⚠️  {channel}")
        print(f"   Missing required fields: {', '.join(unfilled)}")
    else:
        print(f"\n✅ {channel}")
        print(f"   All required fields filled")
        print(f"   Calibration date: {template.get('calibration_date')}")
        print(f"   Gain: {template.get('gain_correction')}, Sa: {template.get('sa_correction')}")

if not all_complete:
    print("\n" + "=" * 80)
    print("⚠️  WARNING: Some templates have missing required fields.")
    print("Fill in the missing values before using the calibration data.")
else:
    print("\n" + "=" * 80)
    print("✅ All calibration templates are complete!")

Loaded 5 calibration templates

⚠️  WBT 400479-15 ES18_ES
   Missing required fields: gain_correction, sa_correction, equivalent_beam_angle, absorption_indicative, sound_speed_indicative

⚠️  WBT 400503-15 ES70-7C_ES
   Missing required fields: gain_correction, sa_correction, equivalent_beam_angle, absorption_indicative, sound_speed_indicative

⚠️  WBT 400509-15 ES200-7C_ES
   Missing required fields: gain_correction, sa_correction, equivalent_beam_angle, absorption_indicative, sound_speed_indicative

⚠️  WBT 400517-15 ES120-7C_ES
   Missing required fields: gain_correction, sa_correction, equivalent_beam_angle, absorption_indicative, sound_speed_indicative

⚠️  WBT 400528-15 ES38-7_ES
   Missing required fields: gain_correction, sa_correction, equivalent_beam_angle, absorption_indicative, sound_speed_indicative

⚠️  WARNING: Some templates have missing required fields.
Fill in the missing values before using the calibration data.


In [9]:
# =============================================================================
# STEP 6 (continued): UPDATE CALIBRATION CONFIGURATIONS FROM TEMPLATES
# =============================================================================
# Regenerate the calibration_configurations.yaml file from the individual 
# template files to ensure both representations are in sync.

# Update the calibration dict from loaded templates
calibration_dict = loaded_templates

# Save updated calibration configurations WITH comments
# (reuse the function from Step 4)
save_multi_channel_config_with_comments(calibration_dict, CALIBRATION_DICT_PATH)

print(f"✅ Updated calibration configurations: {CALIBRATION_DICT_PATH}")
print(f"   (Synced from {len(loaded_templates)} individual template files)")

✅ Updated calibration configurations: Manual_Pipeline_Outputs\mapping_files\calibration_configurations.yaml
   (Synced from 5 individual template files)


In [10]:
# =============================================================================
# STEP 7: EXAMPLE USAGE - RETRIEVE CALIBRATION FOR RAW FILE + CHANNEL
# =============================================================================
# Demonstrates how to use the mapping files to retrieve calibration data
# for a specific raw file and channel combination.
#
# Function used: get_calibration (from mapping_algorithm)

print("=" * 80)
print("EXAMPLE: RETRIEVING CALIBRATION DATA")
print("=" * 80)

# Get the first file and first channel as an example
if mapping_dict:
    example_filename = list(mapping_dict.keys())[0]
    if mapping_dict[example_filename]:
        example_channel = list(mapping_dict[example_filename].keys())[0]
        
        # Retrieve calibration using the in-memory dictionaries
        cal_data = get_calibration(example_filename, example_channel, mapping_dict, calibration_dict)
        
        if cal_data:
            print(f"\nCalibration for '{example_filename}' -> '{example_channel}':")
            print(f"  Calibration date: {cal_data.get('calibration_date')}")
            print(f"  Frequency: {cal_data.get('frequency')}")
            print(f"  Gain correction: {cal_data.get('gain_correction')}")
            print(f"  Sa correction: {cal_data.get('sa_correction')}")
            print(f"  Equivalent beam angle: {cal_data.get('equivalent_beam_angle')}")
            print(f"  Sound speed: {cal_data.get('sound_speed_indicative')}")
            print(f"  Absorption: {cal_data.get('absorption_indicative')}")
        else:
            print(f"No calibration found for {example_filename} -> {example_channel}")

EXAMPLE: RETRIEVING CALIBRATION DATA

Calibration for 'D20241004-T061522.raw' -> 'WBT 400479-15 ES18_ES':
  Calibration date: 2026-02-18
  Frequency: [18000.0]
  Gain correction: [None]
  Sa correction: [None]
  Equivalent beam angle: None
  Sound speed: None
  Absorption: None


In [11]:
# =============================================================================
# PIPELINE SUMMARY
# =============================================================================

print("=" * 80)
print("MANUAL PIPELINE COMPLETE - OUTPUT FILES SUMMARY")
print("=" * 80)

def list_files_recursive(folder, indent=0):
    """List all files in a folder recursively."""
    if not folder.exists():
        return
    for item in sorted(folder.iterdir()):
        if item.is_dir():
            print("  " * indent + f"📁 {item.name}/")
            list_files_recursive(item, indent + 1)
        else:
            size_kb = item.stat().st_size / 1024
            print("  " * indent + f"📄 {item.name} ({size_kb:.1f} KB)")

print(f"\nOutput directory: {OUTPUT_BASE}")
print("-" * 40)
list_files_recursive(OUTPUT_BASE)

print("\n" + "=" * 80)
print("OUTPUT FILE DESCRIPTIONS")
print("=" * 80)
print("""
1. raw_file_configs/raw_file_configs.yaml
   - Channel configurations extracted from each raw file
   - Useful for understanding what channels/frequencies are in your data

2. mapping_files/channel_to_calibration_mapping.yaml
   - Maps each raw file + channel to a calibration configuration key
   - Used to look up which calibration to use for each channel

3. mapping_files/calibration_configurations.yaml
   - Contains ALL calibration templates/data in a single file
   - Alternative to editing individual files (synced from templates)

4. Single_Channel_Calibration_Templates/*.yml
   - Individual template files for each unique channel configuration
   - Edit these to fill in your calibration values
""")

print("=" * 80)
print("WORKFLOW SUMMARY")  
print("=" * 80)
print("""
STEP 1-4: Run automatically to extract raw file configs and generate templates
STEP 5:   USER ACTION - Fill in calibration values in template files
STEP 6:   Run to validate and sync calibration data
STEP 7:   Use the mapping to retrieve calibration for any file/channel

For echopype calibration, extract these parameters from the calibration data:
    - gain_correction
    - sa_correction  
    - equivalent_beam_angle
    - sound_speed_indicative
    - absorption_indicative
""")

MANUAL PIPELINE COMPLETE - OUTPUT FILES SUMMARY

Output directory: Manual_Pipeline_Outputs
----------------------------------------
📁 mapping_files/
  📄 calibration_configurations.yaml (13.2 KB)
  📄 channel_to_calibration_mapping.yaml (0.6 KB)
📁 raw_file_configs/
  📄 raw_file_configs.yaml (3.6 KB)
📁 Single_Channel_Calibration_Templates/
  📄 2026-02-18__WBT 400479-15 ES18_ES__NoSN__0__0.001024__1000.0__18000.0__18000.0.yml (7.4 KB)
  📄 2026-02-18__WBT 400503-15 ES70-7C_ES__NoSN__0__0.001024__750.0__70000.0__70000.0.yml (7.4 KB)
  📄 2026-02-18__WBT 400509-15 ES200-7C_ES__NoSN__0__0.001024__150.0__200000.0__200000.0.yml (7.4 KB)
  📄 2026-02-18__WBT 400517-15 ES120-7C_ES__NoSN__0__0.001024__250.0__120000.0__120000.0.yml (7.4 KB)
  📄 2026-02-18__WBT 400528-15 ES38-7_ES__447__0__0.001024__1000.0__38000.0__38000.0.yml (7.4 KB)

OUTPUT FILE DESCRIPTIONS

1. raw_file_configs/raw_file_configs.yaml
   - Channel configurations extracted from each raw file
   - Useful for understanding what channel